# GEO877 Spatial Algorithms Project

**Members:** Javier Feller, Maximilian Lengenfelder, Pascal Andreas Heiniger, Till Huber

**Research question:** How does the spatial distribution of Flickr photo data differ between cities and parks in Germany in terms of clustering, measured with the Nearest Neighbor Index (NNI)?

**Notebook structure**

1. Setup
2. Data Preparation
3. Spatial Algorithms
   - 3.1 Spatial Indexing
   - 3.2 Point in Polygon
   - 3.3 Nearest Neighbor Index
4. Results & Visualisations

GeoPandas is used only for reading GeoJSON files, CRS transformation, area calculation, and plotting support. The spatial indexing, point-in-polygon, and nearest-neighbor algorithms are implemented manually using course-style Python classes.

# 1. Setup

This section imports the permitted libraries, defines paths, and sets a few switches for full-data and sample runs.

In [1]:
import os
from pathlib import Path

PROJECT_DIR = Path.cwd()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(PROCESSED_DIR / "matplotlib_cache"))

import csv
import math
import time
from collections import defaultdict

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from numpy import sqrt, radians, arcsin, sin, cos

# ---------------------------------------------------------------------
# Run switches
# ---------------------------------------------------------------------
USE_SAMPLE = False              # False = full project run. True = quick classroom/debug run.
SAMPLE_MAX_FILES = 12           # only used when USE_SAMPLE is True
SAMPLE_ROWS_PER_FILE = 5000     # only used when USE_SAMPLE is True
SAMPLE_MAX_COMPILED_ROWS = 100000

REBUILD_COMPILED_CACHE = False
REBUILD_CANDIDATE_CACHE = False
REBUILD_PIP_RESULTS = False
REBUILD_NNI_RESULTS = False

# Spatial-index resolutions in degrees. 0.05 deg is about 3.5-5.5 km in Germany.
GLOBAL_INDEX_RESOLUTION = 0.05
NNI_INDEX_RESOLUTION = 0.01
CSV_CHUNK_SIZE = 500000

RUN_SUFFIX = "_sample" if USE_SAMPLE else ""

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
DATA_BASE_DIR = PROJECT_DIR / "data" / "raw" / "DE"
GEOJSON_PARKS = PROJECT_DIR / "data" / "raw" / "park_polygons.geojson"
GEOJSON_CITIES = PROJECT_DIR / "data" / "raw" / "city_polygons.geojson"

COMPILED_CSV = PROCESSED_DIR / f"flickr_points_compiled{RUN_SUFFIX}.csv"
CANDIDATE_CSV = PROCESSED_DIR / f"flickr_points_bbox_candidates{RUN_SUFFIX}.csv"
INSIDE_CSV = PROCESSED_DIR / f"flickr_points_inside_polygons{RUN_SUFFIX}.csv"
NNI_RESULTS_CSV = PROCESSED_DIR / f"nni_results_by_polygon{RUN_SUFFIX}.csv"

EXPECTED_FLICKR_COLUMNS = [
    "ID", "Latitude", "Longitude", "NAME", "URL", "PhotoID", "Owner", "UserID",
    "DateTaken", "UploadDate", "Views", "Tags", "MTags"
]

print(f"Project directory: {PROJECT_DIR}")
print(f"Sample mode: {USE_SAMPLE}")
print(f"Processed files will be written to: {PROCESSED_DIR}")

Matplotlib is building the font cache; this may take a moment.


Project directory: /Users/maxlengenfelder/Desktop/FS26/GEO877/Project/GEO877_Flickr_Project
Sample mode: False
Processed files will be written to: /Users/maxlengenfelder/Desktop/FS26/GEO877/Project/GEO877_Flickr_Project/data/processed


# 2. Data Preparation

The Flickr data are stored as many `.txt` files inside `data/raw/DE` and its subfolders. Every `settings.txt` file is ignored. The valid photo files all use the same CSV header.

The compiled cache is written once and reused in later runs. The full raw dataset has about 14.8 million rows, so later sections process the compiled CSV in chunks and only instantiate algorithm `Point` objects for relevant bounding-box candidates.

In [2]:
def find_flickr_txt_files(base_dir):
    files = sorted(
        path for path in Path(base_dir).rglob("*.txt")
        if path.name.lower() != "settings.txt"
    )
    settings_files = sorted(Path(base_dir).rglob("settings.txt"))
    return files, settings_files


def validate_flickr_headers(files, settings_files, expected_columns):
    header_counts = defaultdict(int)
    for path in files:
        with open(path, "r", encoding="utf-8", errors="replace", newline="") as f:
            reader = csv.reader(f)
            header = tuple(next(reader))
            header_counts[header] += 1

    print(f"Flickr data files found: {len(files)}")
    print(f"settings.txt files excluded: {len(settings_files)}")
    print(f"Header variants found: {len(header_counts)}")

    for header, count in header_counts.items():
        print(f"Header used by {count} files: {header}")
        if list(header) != expected_columns:
            raise ValueError("Unexpected Flickr file header. Inspect the raw text files before continuing.")

    return header_counts


flickr_files, settings_files = find_flickr_txt_files(DATA_BASE_DIR)
header_counts = validate_flickr_headers(flickr_files, settings_files, EXPECTED_FLICKR_COLUMNS)

Flickr data files found: 403
settings.txt files excluded: 90
Header variants found: 1
Header used by 403 files: ('ID', 'Latitude', 'Longitude', 'NAME', 'URL', 'PhotoID', 'Owner', 'UserID', 'DateTaken', 'UploadDate', 'Views', 'Tags', 'MTags')


In [3]:
def compile_raw_flickr_data(files, output_csv, expected_columns):
    if output_csv.exists() and not REBUILD_COMPILED_CACHE:
        print(f"Using cached compiled Flickr data: {output_csv}")
        preview = pd.read_csv(output_csv, nrows=5)
        print(f"Preview rows loaded: {len(preview)}")
        return output_csv

    files_to_read = files[:SAMPLE_MAX_FILES] if USE_SAMPLE else files
    seen_photo_ids = set()
    rows_read = 0
    rows_written = 0
    duplicate_rows = 0
    invalid_coordinate_rows = 0
    started = time.time()

    with open(output_csv, "w", encoding="utf-8", newline="") as out_file:
        wrote_header = False

        for file_number, path in enumerate(files_to_read, start=1):
            nrows = SAMPLE_ROWS_PER_FILE if USE_SAMPLE else None
            df = pd.read_csv(
                path,
                usecols=expected_columns,
                nrows=nrows,
                on_bad_lines="skip",
                low_memory=False
            )
            rows_read += len(df)

            df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce")
            df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")
            valid_coordinate_mask = (
                df["Latitude"].between(-90, 90) &
                df["Longitude"].between(-180, 180)
            )
            invalid_coordinate_rows += int((~valid_coordinate_mask).sum())
            df = df.loc[valid_coordinate_mask].copy()

            df["PhotoID"] = df["PhotoID"].astype(str)

            before_file_dedup = len(df)
            df = df.drop_duplicates(subset="PhotoID", keep="first").copy()
            duplicate_rows += before_file_dedup - len(df)

            duplicate_mask = df["PhotoID"].isin(seen_photo_ids)
            duplicate_rows += int(duplicate_mask.sum())
            df = df.loc[~duplicate_mask].copy()
            seen_photo_ids.update(df["PhotoID"].tolist())

            df.to_csv(out_file, index=False, header=not wrote_header)
            wrote_header = True
            rows_written += len(df)

            if file_number % 25 == 0 or file_number == len(files_to_read):
                elapsed = time.time() - started
                print(f"Processed {file_number}/{len(files_to_read)} files | written rows: {rows_written:,} | elapsed: {elapsed:.1f}s")

            if USE_SAMPLE and rows_read >= SAMPLE_MAX_COMPILED_ROWS:
                print("Sample row limit reached.")
                break

    print("\nCompiled Flickr cache complete")
    print(f"Rows read: {rows_read:,}")
    print(f"Rows written after cleaning/deduplication: {rows_written:,}")
    print(f"Duplicate PhotoID rows skipped: {duplicate_rows:,}")
    print(f"Invalid coordinate rows skipped: {invalid_coordinate_rows:,}")
    print(f"Output: {output_csv}")
    return output_csv


compile_raw_flickr_data(flickr_files, COMPILED_CSV, EXPECTED_FLICKR_COLUMNS)
flickr_preview = pd.read_csv(COMPILED_CSV, nrows=5)
flickr_preview

Processed 25/403 files | written rows: 1,161,120 | elapsed: 9.1s
Processed 50/403 files | written rows: 2,095,451 | elapsed: 22.5s
Processed 75/403 files | written rows: 2,965,150 | elapsed: 38.2s
Processed 100/403 files | written rows: 3,910,663 | elapsed: 63.9s
Processed 125/403 files | written rows: 4,862,477 | elapsed: 93.1s
Processed 150/403 files | written rows: 5,680,643 | elapsed: 122.7s
Processed 175/403 files | written rows: 6,734,259 | elapsed: 161.4s
Processed 200/403 files | written rows: 7,660,503 | elapsed: 210.7s
Processed 225/403 files | written rows: 8,349,072 | elapsed: 262.4s
Processed 250/403 files | written rows: 8,826,406 | elapsed: 302.1s
Processed 275/403 files | written rows: 9,198,059 | elapsed: 341.2s
Processed 300/403 files | written rows: 10,119,235 | elapsed: 386.9s
Processed 325/403 files | written rows: 10,833,032 | elapsed: 436.9s
Processed 350/403 files | written rows: 11,760,081 | elapsed: 492.9s
Processed 375/403 files | written rows: 12,644,894 | e

,ID,Latitude,Longitude,NAME,URL,PhotoID,Owner,UserID,DateTaken,UploadDate,Views,Tags,MTags
0,1,49.037234,7.941784,Wissembourg; Bas-Rhin; le cloître (2),https://farm3.staticflickr.com/2618/3755526678...,3755526678,roger joseph,32305224@N07,7/22/2009 16:00:42,7/25/2009 16:53:04,18,;alsace;églisesgothiques;,NaN
1,2,49.037234,7.941784,Wissembourg; Bas-Rhin (2),https://farm3.staticflickr.com/2493/3754721477...,3754721477,roger joseph,32305224@N07,7/22/2009 15:48:48,7/25/2009 16:51:16,20,;alsace;églisesgothiques;,NaN
2,3,49.037234,7.941784,Wissembourg; Bas-Rhin (3),https://farm4.staticflickr.com/3467/3755522558...,3755522558,roger joseph,32305224@N07,7/22/2009 15:49:56,7/25/2009 16:51:30,26,;alsace;églisesgothiques;,NaN
3,4,49.037234,7.941784,Wissembourg; Bas-Rhin (7),https://farm3.staticflickr.com/2605/3754723413...,3754723413,roger joseph,32305224@N07,7/22/2009 15:52:56,7/25/2009 16:52:01,26,;alsace;églisesgothiques;,NaN
4,5,49.037234,7.941784,Wissembourg; Bas-Rhin (20),https://farm3.staticflickr.com/2435/3755524714...,3755524714,roger joseph,32305224@N07,7/22/2009 16:06:14,7/25/2009 16:52:18,19,;alsace;églisesgothiques;,NaN


In [4]:
def load_polygon_inputs(city_path, park_path):
    polygon_records = []
    area_lookup = {}

    layers = [
        ("City", city_path, "Geografisc"),
        ("Park", park_path, "NAME"),
    ]

    for polygon_type, path, name_field in layers:
        gdf_original = gpd.read_file(path)
        print(f"{polygon_type} source CRS: {gdf_original.crs}")

        # GeoPandas is used only here for CRS transformation and metric area calculation.
        gdf_lonlat = gdf_original.to_crs(epsg=4326)
        gdf_metric = gdf_original.to_crs(epsg=25832)

        minx, miny, maxx, maxy = gdf_lonlat.total_bounds
        print(f"{polygon_type} transformed bounds EPSG:4326: {(round(minx, 5), round(miny, 5), round(maxx, 5), round(maxy, 5))}")

        for idx, row in gdf_lonlat.iterrows():
            geom = row.geometry
            name_value = row[name_field] if name_field in row and pd.notna(row[name_field]) else f"{polygon_type}_{idx}"
            polygon_name = f"{polygon_type}_{name_value}"
            area_m2 = float(gdf_metric.geometry.iloc[idx].area)
            area_lookup[polygon_name] = area_m2

            if geom.geom_type == "Polygon":
                parts = [geom]
            elif geom.geom_type == "MultiPolygon":
                parts = list(geom.geoms)
            else:
                continue

            for part_number, part in enumerate(parts):
                exterior = [(float(x), float(y)) for x, y in part.exterior.coords]
                holes = [
                    [(float(x), float(y)) for x, y in interior.coords]
                    for interior in part.interiors
                ]
                polygon_records.append({
                    "name": polygon_name,
                    "type": polygon_type,
                    "part": part_number,
                    "exterior": exterior,
                    "holes": holes,
                    "area_m2": area_m2,
                    "geometry": part,
                })

    return polygon_records, area_lookup


polygon_records, polygon_area_lookup = load_polygon_inputs(GEOJSON_CITIES, GEOJSON_PARKS)
print(f"Prepared polygon parts for manual algorithms: {len(polygon_records)}")
print(f"Polygons with holes: {sum(1 for p in polygon_records if p['holes'])}")

City source CRS: EPSG:25832
City transformed bounds EPSG:4326: (np.float64(6.68982), np.float64(48.06155), np.float64(13.76047), np.float64(53.73847))
Park source CRS: EPSG:4326
Park transformed bounds EPSG:4326: (np.float64(6.58083), np.float64(47.5324), np.float64(13.83964), np.float64(55.09917))
Prepared polygon parts for manual algorithms: 20
Polygons with holes: 4


# 3. Spatial Algorithms

The next cells define the course-style spatial classes. The classes deliberately avoid GeoPandas/Shapely spatial operations: points, bounding boxes, polygon containment, indexing, and nearest-neighbor calculations are all computed manually.

In [5]:
class Point():
    # initialise
    def __init__(self, x=None, y=None, pid=None, attrs=None):
        self.x = float(x)  # longitude
        self.y = float(y)  # latitude
        self.id = pid
        self.attributes = attrs or {}

    # representation
    def __repr__(self):
        return f"Point(x={self.x}, y={self.y})"

    # test for equality between two points, following the course examples
    def __eq__(self, other):
        if not isinstance(other, Point):
            return NotImplemented
        return self.x == other.x and self.y == other.y

    def isEqual(self, other):
        return self.x == other.x and self.y == other.y

    def __hash__(self):
        return hash((self.x, self.y))

    # calculate Euclidean distance between two points
    def distEuclidean(self, other):
        return sqrt((self.x - other.x)**2 + (self.y - other.y)**2)

    # calculate Manhattan distance between two points
    def distManhattan(self, other):
        return abs(self.x - other.x) + abs(self.y - other.y)

    # Haversine distance between two lon/lat points on a sphere, returned in metres
    def distHaversine(self, other):
        r = 6371000
        phi1 = radians(self.y)
        phi2 = radians(other.y)
        lam1 = radians(self.x)
        lam2 = radians(other.x)

        d = 2 * r * arcsin(sqrt(
            sin((phi2 - phi1) / 2)**2 +
            cos(phi1) * cos(phi2) * sin((lam2 - lam1) / 2)**2
        ))
        return float(d)

    # determine position of this point in relation to a vector of two other points
    def sideLine(self, p1, p2):
        side = int((p2.x - p1.x) * (self.y - p1.y) - (self.x - p1.x) * (p2.y - p1.y))
        if side != 0:
            side = side / abs(side)
        return side


class Bbox():
    # initialise
    def __init__(self, data):
        if isinstance(data, Segment):
            x = [data.start.x, data.end.x]
            y = [data.start.y, data.end.y]
        else:
            x = [p.x for p in data]
            y = [p.y for p in data]

        self.ll = Point(min(x), min(y))
        self.ur = Point(max(x), max(y))
        self.ctr = Point((min(x) + max(x)) / 2, (min(y) + max(y)) / 2)
        self.area = abs(max(x) - min(x)) * abs(max(y) - min(y))

    def __repr__(self):
        return f"Bounding box with lower-left {self.ll} and upper-right {self.ur}"

    def testOverlap(self, other):
        if (self.ur.x >= other.ll.x and other.ur.x >= self.ll.x and
            self.ur.y >= other.ll.y and other.ur.y >= self.ll.y):
            return True
        return False

    def containsPoint(self, p):
        if (self.ur.x >= p.x >= self.ll.x and self.ur.y >= p.y >= self.ll.y):
            return True
        return False

    def intersects(self, other):
        if (self.ur.x >= other.ll.x and other.ur.x >= self.ll.x and
            self.ur.y >= other.ll.y and other.ur.y >= self.ll.y):
            return True
        return False

    def intersectsRegion(self, other):
        if not self.intersects(other):
            return None
        llx = max(self.ll.x, other.ll.x)
        lly = max(self.ll.y, other.ll.y)
        urx = min(self.ur.x, other.ur.x)
        ury = min(self.ur.y, other.ur.y)
        return Bbox([Point(llx, lly), Point(urx, ury)])


class Segment():
    # initialise
    def __init__(self, p0, p1, sid=None):
        self.start = p0
        self.end = p1
        self.sid = sid
        self.length = p0.distEuclidean(p1)

    def __repr__(self):
        return f"Segment with start {self.start} and end {self.end}."

    def isIdentical(self, other):
        if (self.start.isEqual(other.start) or self.start.isEqual(other.end)) and \
           (self.end.isEqual(other.end) or self.end.isEqual(other.start)):
            return True
        return False

    def intersects(self, other):
        self_bbox = Bbox(self)
        other_bbox = Bbox(other)
        bbox_overlap = self_bbox.testOverlap(other_bbox)

        if bbox_overlap == False:
            return False

        apq = self.start.sideLine(other.start, other.end)
        bpq = self.end.sideLine(other.start, other.end)
        pab = other.start.sideLine(self.start, self.end)
        qab = other.end.sideLine(self.start, self.end)

        if (apq + bpq == 0 and pab + qab == 0):
            return True
        return False


class Polygon():
    # child-style polygon class adapted from the course Point-in-Polygon notebook
    def __init__(self, exterior=None, holes=None, name=None, poly_type=None, area_m2=None):
        self.points = [Point(x, y) for x, y in exterior]
        if self.points[0] != self.points[-1]:
            self.points.append(self.points[0])

        self.holes = []
        for ring in holes or []:
            hole_points = [Point(x, y) for x, y in ring]
            if hole_points and hole_points[0] != hole_points[-1]:
                hole_points.append(hole_points[0])
            self.holes.append(hole_points)

        self.size = len(self.points)
        self.name = name
        self.poly_type = poly_type
        self.area_m2 = area_m2
        self.bbox = Bbox(self.points)

    def __repr__(self):
        return f"Polygon(name={self.name}, type={self.poly_type}, points={self.size}, holes={len(self.holes)})"

    def __getitem__(self, key):
        return self.points[key]

    def isClosed(self):
        return self.points[0] == self.points[-1]

    def _ringContainsPoint(self, p, ring_points):
        # Ray-casting test from the point-in-polygon practical, implemented with direct arithmetic
        # to avoid creating millions of temporary Segment objects for the large Flickr dataset.
        count = 0
        for i in range(0, len(ring_points) - 1):
            start = ring_points[i]
            end = ring_points[i + 1]

            if (p.y > min(start.y, end.y)):
                if (p.y <= max(start.y, end.y)):
                    if (p.x <= max(start.x, end.x)):
                        if (start.y != end.y):
                            x_intersection = start.x + (p.y - start.y) * (end.x - start.x) / (end.y - start.y)
                            if p.x <= x_intersection:
                                count += 1

        return count % 2 != 0

    def containsPoint(self, p):
        if self.bbox.containsPoint(p) == False:
            return False

        if self._ringContainsPoint(p, self.points) == False:
            return False

        # Interior rings are holes. A point inside a hole is not inside the polygon.
        for hole in self.holes:
            if self._ringContainsPoint(p, hole):
                return False

        return True

In [6]:
def build_course_polygons(polygon_records):
    polygons = []
    for record in polygon_records:
        polygons.append(Polygon(
            exterior=record["exterior"],
            holes=record["holes"],
            name=record["name"],
            poly_type=record["type"],
            area_m2=record["area_m2"]
        ))
    return polygons


course_polygons = build_course_polygons(polygon_records)
print(f"Manual polygon objects created: {len(course_polygons)}")
print(course_polygons[0])

# Small point-in-polygon test from the course style examples
sample_polygon = Polygon(
    exterior=[[0, 0], [10, 0], [10, 10], [0, 10], [0, 0]],
    holes=[[[3, 3], [7, 3], [7, 7], [3, 7], [3, 3]]],
    name="Sample_with_hole",
    poly_type="Test",
    area_m2=100
)
assert sample_polygon.containsPoint(Point(2, 2)) == True
assert sample_polygon.containsPoint(Point(5, 5)) == False
assert sample_polygon.containsPoint(Point(12, 2)) == False
print("PIP test with exterior and hole passed.")

Manual polygon objects created: 20
Polygon(name=City_Hamburg, type=City, points=1227, holes=0)
PIP test with exterior and hole passed.


## 3.1 Spatial Indexing

The grid index follows the practical 6 idea: points are assigned to uniform cells, and region queries only inspect cells overlapping a query bounding box.

Before building the point index, the compiled Flickr CSV is filtered in chunks against the polygon bounding boxes. This avoids creating Python objects for all 14.8 million raw rows.

In [7]:
class PointIndex():
    # initialise the index
    def __init__(self, data, box=None, res=0.05):
        self.res = res
        self.input_count = len(data)
        self.bBox = box if box else Bbox(data)

        w = self.bBox.ur.x - self.bBox.ll.x
        h = self.bBox.ur.y - self.bBox.ll.y
        self.nCols = int(w / self.res) + 1
        self.nRows = int(h / self.res) + 1

        ur = Point(
            self.bBox.ll.x + (self.nCols * self.res),
            self.bBox.ll.y + (self.nRows * self.res)
        )
        ll = self.bBox.ll
        self.bBox = Bbox([ll, ur])
        self.maxIndex = (self.nCols * self.nRows) - 1
        self.points = [[0, []] for _ in range(self.maxIndex + 1)]
        self.bigArray = []

        self.addPoints(data)

    def __repr__(self):
        return f"PointIndex(res={self.res}, nCols={self.nCols}, nRows={self.nRows}, points={len(self.bigArray)})"

    def addPoints(self, data):
        for p in data:
            self.addPoint(p)
            self.bigArray.append(p)

    def addPoint(self, p):
        i = self.pointIndex(p)
        if 0 <= i <= self.maxIndex:
            self.points[i][0] += 1
            self.points[i][1].append(p)

    def pointIndex(self, p):
        j = int((p.y - self.bBox.ll.y) / self.res)
        i = int((p.x - self.bBox.ll.x) / self.res)
        return (j * self.nCols) + i

    def regionQuery(self, region):
        query = self.bBox.intersectsRegion(region)
        if query is None:
            return [], 0

        c_start = max(0, int((query.ll.x - self.bBox.ll.x) / self.res))
        c_end = min(self.nCols - 1, int((query.ur.x - self.bBox.ll.x) / self.res))
        r_start = max(0, int((query.ll.y - self.bBox.ll.y) / self.res))
        r_end = min(self.nRows - 1, int((query.ur.y - self.bBox.ll.y) / self.res))

        ps = []
        for r in range(r_start, r_end + 1):
            for c in range(c_start, c_end + 1):
                idx = (r * self.nCols) + c
                if 0 <= idx <= self.maxIndex and self.points[idx][0] > 0:
                    ps.extend(self.points[idx][1])

        final = []
        for p in ps:
            if region.containsPoint(p):
                final.append(p)

        return final, len(final)

    def bruteRegionQuery(self, region):
        final = []
        for p in self.bigArray:
            if region.containsPoint(p):
                final.append(p)
        return final, len(final)

    def nearestPoint(self, p, method="haversine"):
        step = self.res
        max_step = max(self.bBox.ur.x - self.bBox.ll.x, self.bBox.ur.y - self.bBox.ll.y) + self.res
        candidates = []

        while len(candidates) == 0 and step <= max_step:
            search_box = Bbox([Point(p.x - step, p.y - step), Point(p.x + step, p.y + step)])
            candidates, count = self.regionQuery(search_box)
            candidates = [q for q in candidates if q is not p]
            step += self.res

        if len(candidates) == 0:
            return None, None

        nearest_point, nearest_distance = self.__minDist(candidates, p, method=method)

        if method == "haversine":
            lat_buffer = nearest_distance / 111320.0
            lon_scale = max(math.cos(math.radians(p.y)), 0.01)
            lon_buffer = nearest_distance / (111320.0 * lon_scale)
        else:
            lat_buffer = nearest_distance
            lon_buffer = nearest_distance

        refined_box = Bbox([Point(p.x - lon_buffer, p.y - lat_buffer), Point(p.x + lon_buffer, p.y + lat_buffer)])
        refined_candidates, refined_count = self.regionQuery(refined_box)
        refined_candidates = [q for q in refined_candidates if q is not p]

        if len(refined_candidates) > len(candidates):
            nearest_point, nearest_distance = self.__minDist(refined_candidates, p, method=method)

        return nearest_point, nearest_distance

    def __minDist(self, points, p, method="haversine"):
        best_point = None
        best_distance = float("inf")
        for q in points:
            if method == "haversine":
                d = p.distHaversine(q)
            else:
                d = p.distEuclidean(q)
            if d < best_distance:
                best_point = q
                best_distance = d
        return best_point, best_distance

In [8]:
def write_bbox_candidates(compiled_csv, output_csv, polygons):
    if output_csv.exists() and not REBUILD_CANDIDATE_CACHE:
        print(f"Using cached bounding-box candidates: {output_csv}")
        return output_csv

    polygon_bboxes = [(poly.bbox.ll.x, poly.bbox.ll.y, poly.bbox.ur.x, poly.bbox.ur.y) for poly in polygons]
    total_rows = 0
    total_candidates = 0
    started = time.time()

    with open(output_csv, "w", encoding="utf-8", newline="") as out_file:
        wrote_header = False
        for chunk_number, chunk in enumerate(pd.read_csv(compiled_csv, chunksize=CSV_CHUNK_SIZE), start=1):
            total_rows += len(chunk)
            lons = chunk["Longitude"].to_numpy(dtype=float)
            lats = chunk["Latitude"].to_numpy(dtype=float)
            candidate_mask = np.zeros(len(chunk), dtype=bool)

            for minx, miny, maxx, maxy in polygon_bboxes:
                candidate_mask |= ((lons >= minx) & (lons <= maxx) & (lats >= miny) & (lats <= maxy))

            candidates = chunk.loc[candidate_mask].copy()
            total_candidates += len(candidates)
            candidates.to_csv(out_file, index=False, header=not wrote_header)
            wrote_header = True

            if chunk_number % 10 == 0:
                elapsed = time.time() - started
                print(f"Chunks processed: {chunk_number} | rows scanned: {total_rows:,} | candidates: {total_candidates:,} | elapsed: {elapsed:.1f}s")

            if USE_SAMPLE and total_rows >= SAMPLE_MAX_COMPILED_ROWS:
                break

    print(f"Bounding-box candidate cache written: {output_csv}")
    print(f"Rows scanned: {total_rows:,}")
    print(f"Candidate rows: {total_candidates:,}")
    return output_csv


write_bbox_candidates(COMPILED_CSV, CANDIDATE_CSV, course_polygons)
candidate_preview = pd.read_csv(CANDIDATE_CSV, nrows=5)
candidate_preview

Chunks processed: 10 | rows scanned: 5,000,000 | candidates: 1,331,573 | elapsed: 14.3s
Chunks processed: 20 | rows scanned: 10,000,000 | candidates: 3,497,601 | elapsed: 31.4s


/var/folders/dn/y1kk3fj96gq2w74dh77lmghw0000gn/T/ipykernel_7909/536938506.py:13: DtypeWarning: Columns (0: PhotoID) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk_number, chunk in enumerate(pd.read_csv(compiled_csv, chunksize=CSV_CHUNK_SIZE), start=1):
/var/folders/dn/y1kk3fj96gq2w74dh77lmghw0000gn/T/ipykernel_7909/536938506.py:13: DtypeWarning: Columns (0: Views) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk_number, chunk in enumerate(pd.read_csv(compiled_csv, chunksize=CSV_CHUNK_SIZE), start=1):
/var/folders/dn/y1kk3fj96gq2w74dh77lmghw0000gn/T/ipykernel_7909/536938506.py:13: DtypeWarning: Columns (0: PhotoID, 1: Views) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk_number, chunk in enumerate(pd.read_csv(compiled_csv, chunksize=CSV_CHUNK_SIZE), start=1):
/var/folders/dn/y1kk3fj96gq2w74dh77lmghw0000gn/T/ipykernel_7909/536938506.py:13: DtypeWarning: Columns (0: PhotoID) 

Bounding-box candidate cache written: /Users/maxlengenfelder/Desktop/FS26/GEO877/Project/GEO877_Flickr_Project/data/processed/flickr_points_bbox_candidates.csv
Rows scanned: 13,649,294
Candidate rows: 3,497,601


,ID,Latitude,Longitude,NAME,URL,PhotoID,Owner,UserID,DateTaken,UploadDate,Views,Tags,MTags
0,300,49.022423,8.170961,Halloween,https://farm4.staticflickr.com/3947/1539291677...,15392916777,Frank&Devil,112897506@N05,10/19/2014 17:36:24,10/20/2014 1:38:10,4258,;;,NaN
1,301,49.022423,8.170961,Dahlie,https://farm4.staticflickr.com/3935/1539313098...,15393130980,Frank&Devil,112897506@N05,10/19/2014 17:27:25,10/20/2014 1:04:19,4436,;;,NaN
2,337,49.023461,8.195800,Besökare Samtidigt,https://farm4.staticflickr.com/3919/1415134443...,14151344438,Photostream from the Meanwhile Exhibition.,100597270@N04,6/03/2014 17:08:12,6/03/2014 15:08:12,20,;;,NaN
3,353,49.019296,8.248500,Milder Milchling (Lactarius mitissimus),https://farm4.staticflickr.com/3830/1004239040...,10042390404,Smaddin,100196911@N06,10/01/2013 17:53:13,10/01/2013 19:22:18,141,;macro;canon;eos;mashroom;moos;pilz;500d;,NaN
4,354,49.019296,8.248500,Violetter Lacktrichterling (Laccaria amethystea),https://farm8.staticflickr.com/7331/1004253721...,10042537213,Smaddin,100196911@N06,10/01/2013 18:04:25,10/01/2013 19:22:18,197,;macro;canon;eos;lila;pilz;violett;500d;,NaN


In [9]:
def point_from_row(row):
    attrs = {col: getattr(row, col) for col in EXPECTED_FLICKR_COLUMNS if hasattr(row, col)}
    return Point(row.Longitude, row.Latitude, pid=row.PhotoID, attrs=attrs)


def load_candidate_points(candidate_csv):
    df = pd.read_csv(candidate_csv)
    points = [point_from_row(row) for row in df.itertuples(index=False)]
    return df, points


candidate_df, spatial_points = load_candidate_points(CANDIDATE_CSV)
print(f"Candidate rows loaded for manual algorithms: {len(candidate_df):,}")

if len(spatial_points) == 0:
    raise ValueError("No candidate points found. Check coordinate systems and polygon bounds.")

global_grid = PointIndex(spatial_points, res=GLOBAL_INDEX_RESOLUTION)
print(global_grid)

# Verify indexed region query against brute-force query on a small subset.
subset_points = spatial_points[:min(5000, len(spatial_points))]
subset_index = PointIndex(subset_points, res=GLOBAL_INDEX_RESOLUTION)
test_region = course_polygons[0].bbox
indexed_points, indexed_count = subset_index.regionQuery(test_region)
brute_points, brute_count = subset_index.bruteRegionQuery(test_region)
assert indexed_count == brute_count
print(f"Spatial index test passed: {indexed_count} indexed candidates equals brute force count.")

Candidate rows loaded for manual algorithms: 3,497,601
PointIndex(res=0.05, nCols=146, nRows=152, points=3497601)
Spatial index test passed: 0 indexed candidates equals brute force count.


## 3.2 Point in Polygon

Each polygon first queries the point index with its bounding box. Only those candidate points are passed into the manual ray-casting point-in-polygon method. Points in park holes are excluded. A Flickr photo can appear once per matching polygon, so overlaps between city and park polygons are preserved independently.

In [10]:
def run_point_in_polygon(polygons, point_index, output_csv):
    if output_csv.exists() and not REBUILD_PIP_RESULTS:
        print(f"Using cached PIP results: {output_csv}")
        return pd.read_csv(output_csv)

    records = []
    started = time.time()

    for polygon_number, poly in enumerate(polygons, start=1):
        candidates, candidate_count = point_index.regionQuery(poly.bbox)
        inside_count = 0

        for p in candidates:
            if poly.containsPoint(p):
                record = p.attributes.copy()
                record["polygon_name"] = poly.name
                record["polygon_type"] = poly.poly_type
                record["polygon_area_m2"] = poly.area_m2
                records.append(record)
                inside_count += 1

        elapsed = time.time() - started
        print(
            f"{polygon_number:02d}/{len(polygons)} {poly.name}: "
            f"bbox candidates={candidate_count:,}, inside={inside_count:,}, elapsed={elapsed:.1f}s"
        )

    output_columns = EXPECTED_FLICKR_COLUMNS + ["polygon_name", "polygon_type", "polygon_area_m2"]
    matched_df = pd.DataFrame(records, columns=output_columns)
    matched_df.to_csv(output_csv, index=False)
    print(f"\nPIP result rows written: {len(matched_df):,}")
    print(f"Output: {output_csv}")
    return matched_df


matched_points_df = run_point_in_polygon(course_polygons, global_grid, INSIDE_CSV)
matched_points_df.head()

01/20 City_Hamburg: bbox candidates=410,932, inside=357,867, elapsed=60.2s
02/20 City_Bremen: bbox candidates=79,735, inside=52,322, elapsed=68.9s
03/20 City_Düsseldorf: bbox candidates=120,566, inside=114,147, elapsed=76.1s
04/20 City_Köln: bbox candidates=228,583, inside=213,065, elapsed=88.0s
05/20 City_Dortmund: bbox candidates=53,965, inside=48,377, elapsed=90.7s
06/20 City_Frankfurt am Main: bbox candidates=244,613, inside=205,253, elapsed=102.6s
07/20 City_Stuttgart: bbox candidates=140,113, inside=128,815, elapsed=111.0s
08/20 City_München: bbox candidates=350,351, inside=337,843, elapsed=130.8s
09/20 City_Berlin: bbox candidates=1,106,229, inside=1,082,508, elapsed=270.7s
10/20 City_Leipzig: bbox candidates=95,401, inside=90,512, elapsed=276.2s
11/20 Park_Schleswig-Holsteinisches Wattenmeer: bbox candidates=44,874, inside=4,923, elapsed=297.2s
12/20 Park_Niedersächsisches Wattenmeer: bbox candidates=53,381, inside=6,318, elapsed=327.5s
13/20 Park_Bayerischer Wald: bbox candida

,ID,Latitude,Longitude,NAME,URL,PhotoID,Owner,UserID,DateTaken,UploadDate,Views,Tags,MTags,polygon_name,polygon_type,polygon_area_m2
0,41042,53.430775,9.919569,Zentriert,https://farm8.staticflickr.com/7067/7035036113...,7035036113,fotoscheibe,51337247@N08,3/31/2012 19:35:30,4/01/2012 13:48:49,266,;deutschland;sonnenuntergang;hamburg;feld;gege...,NaN,City_Hamburg,City,7.365237e+08
1,41043,53.430775,9.919569,Flare,https://farm8.staticflickr.com/7210/7034969235...,7034969235,fotoscheibe,51337247@N08,3/31/2012 19:29:12,4/01/2012 13:20:30,163,;tree;deutschland;sonnenuntergang;hamburg;laub...,NaN,City_Hamburg,City,7.365237e+08
2,41044,53.430775,9.919569,Schatten,https://farm8.staticflickr.com/7080/6888874024...,6888874024,fotoscheibe,51337247@N08,3/31/2012 19:26:30,4/01/2012 13:19:58,298,;tree;deutschland;sonnenuntergang;hamburg;laub...,NaN,City_Hamburg,City,7.365237e+08
3,41045,53.430775,9.919569,Gegenlicht,https://farm8.staticflickr.com/7101/6888869976...,6888869976,fotoscheibe,51337247@N08,3/31/2012 19:24:35,4/01/2012 13:18:08,226,;tree;deutschland;sonnenuntergang;hamburg;laub...,NaN,City_Hamburg,City,7.365237e+08
4,41046,53.430775,9.919569,Dem Licht entgegen.,https://farm8.staticflickr.com/7101/6888866196...,6888866196,fotoscheibe,51337247@N08,3/31/2012 19:24:14,4/01/2012 13:16:29,150,;tree;deutschland;sonnenuntergang;hamburg;laub...,NaN,City_Hamburg,City,7.365237e+08


In [11]:
if len(matched_points_df) == 0:
    print("No Flickr points were found inside the selected polygons.")
else:
    point_counts = (
        matched_points_df
        .groupby(["polygon_type", "polygon_name"])
        .size()
        .reset_index(name="point_count")
        .sort_values(["polygon_type", "point_count"], ascending=[True, False])
    )
    display(point_counts)

,polygon_type,polygon_name,point_count
0,City,City_Berlin,1082508
5,City,City_Hamburg,357867
8,City,City_München,337843
6,City,City_Köln,213065
4,City,City_Frankfurt am Main,205253
9,City,City_Stuttgart,128815
3,City,City_Düsseldorf,114147
7,City,City_Leipzig,90512
1,City,City_Bremen,52322
2,City,City_Dortmund,48377


## 3.3 Nearest Neighbor Index

For each polygon, matched Flickr points are grouped and indexed again with a local grid. The observed mean nearest-neighbor distance uses the Haversine method from the distance practical. The expected distance uses the standard random point pattern formula:

\[
E(d) = 
rac{1}{2\sqrt{n/A}}
\]

where `n` is the number of points and `A` is the polygon area in square meters.

In [12]:
def dataframe_to_points(df):
    points = []
    for row in df.itertuples(index=False):
        attrs = {col: getattr(row, col) for col in EXPECTED_FLICKR_COLUMNS if hasattr(row, col)}
        points.append(Point(row.Longitude, row.Latitude, pid=row.PhotoID, attrs=attrs))
    return points


def compute_nni(points_list, true_area_sq_meters, index_resolution=NNI_INDEX_RESOLUTION):
    n = len(points_list)
    if n < 2 or true_area_sq_meters <= 0:
        return None

    local_index = PointIndex(points_list, res=index_resolution)
    nearest_distances = []

    for i, p in enumerate(points_list, start=1):
        nearest_point, nearest_distance = local_index.nearestPoint(p, method="haversine")
        if nearest_distance is not None:
            nearest_distances.append(nearest_distance)

        if i % 50000 == 0:
            print(f"  nearest-neighbor distances computed: {i:,}/{n:,}")

    if len(nearest_distances) == 0:
        return None

    observed_mean_distance = sum(nearest_distances) / len(nearest_distances)
    point_density = n / true_area_sq_meters
    expected_mean_distance = 1.0 / (2.0 * math.sqrt(point_density))
    nni_value = observed_mean_distance / expected_mean_distance

    return {
        "point_count": n,
        "observed_mean_distance_m": observed_mean_distance,
        "expected_mean_distance_m": expected_mean_distance,
        "nni": nni_value,
    }


def run_nni(matched_df, output_csv):
    result_columns = [
        "polygon_type", "polygon_name", "point_count", "area_m2",
        "observed_mean_distance_m", "expected_mean_distance_m", "nni", "pattern"
    ]

    if output_csv.exists() and not REBUILD_NNI_RESULTS:
        print(f"Using cached NNI results: {output_csv}")
        return pd.read_csv(output_csv)

    if len(matched_df) == 0:
        empty_results = pd.DataFrame(columns=result_columns)
        empty_results.to_csv(output_csv, index=False)
        print(f"No matched points available for NNI. Empty results written: {output_csv}")
        return empty_results

    results = []
    grouped = matched_df.groupby(["polygon_type", "polygon_name"], sort=True)

    for (polygon_type, polygon_name), group in grouped:
        print(f"Computing NNI for {polygon_name} ({len(group):,} points)")
        points = dataframe_to_points(group)
        area_m2 = float(group["polygon_area_m2"].iloc[0])
        nni_stats = compute_nni(points, area_m2)

        if nni_stats is None:
            results.append({
                "polygon_type": polygon_type,
                "polygon_name": polygon_name,
                "point_count": len(group),
                "area_m2": area_m2,
                "observed_mean_distance_m": np.nan,
                "expected_mean_distance_m": np.nan,
                "nni": np.nan,
                "pattern": "not enough points",
            })
            continue

        if nni_stats["nni"] < 1:
            pattern = "clustered"
        elif nni_stats["nni"] > 1:
            pattern = "dispersed"
        else:
            pattern = "random"

        results.append({
            "polygon_type": polygon_type,
            "polygon_name": polygon_name,
            "point_count": nni_stats["point_count"],
            "area_m2": area_m2,
            "observed_mean_distance_m": nni_stats["observed_mean_distance_m"],
            "expected_mean_distance_m": nni_stats["expected_mean_distance_m"],
            "nni": nni_stats["nni"],
            "pattern": pattern,
        })

    results_df = pd.DataFrame(results, columns=result_columns).sort_values(["polygon_type", "nni"])
    results_df.to_csv(output_csv, index=False)
    print(f"NNI results written: {output_csv}")
    return results_df


nni_results = run_nni(matched_points_df, NNI_RESULTS_CSV)
nni_results

Computing NNI for City_Berlin (1,082,508 points)
  nearest-neighbor distances computed: 50,000/1,082,508
  nearest-neighbor distances computed: 100,000/1,082,508
  nearest-neighbor distances computed: 150,000/1,082,508


KeyboardInterrupt: 

# 4. Results & Visualisations

The summary compares city and park NNI values. NNI below 1 indicates clustering; values above 1 indicate dispersion relative to a random point pattern.

In [ ]:
if len(nni_results) == 0:
    print("No NNI results available.")
else:
    summary = (
        nni_results
        .dropna(subset=["nni"])
        .groupby("polygon_type")
        .agg(
            polygons=("polygon_name", "count"),
            total_points=("point_count", "sum"),
            mean_nni=("nni", "mean"),
            median_nni=("nni", "median"),
            mean_observed_distance_m=("observed_mean_distance_m", "mean"),
            mean_expected_distance_m=("expected_mean_distance_m", "mean"),
        )
        .reset_index()
    )
    display(summary)

    if set(summary["polygon_type"]) >= {"City", "Park"}:
        avg_city = float(summary.loc[summary["polygon_type"] == "City", "mean_nni"].iloc[0])
        avg_park = float(summary.loc[summary["polygon_type"] == "Park", "mean_nni"].iloc[0])
        stronger_cluster = "cities" if avg_city < avg_park else "parks"
        print(f"Average city NNI: {avg_city:.4f}")
        print(f"Average park NNI: {avg_park:.4f}")
        print(f"The Flickr photo pattern is more clustered in {stronger_cluster} based on the lower average NNI.")

In [ ]:
if len(nni_results.dropna(subset=["nni"])) > 0:
    plot_df = nni_results.dropna(subset=["nni"]).sort_values("nni")
    colors = plot_df["polygon_type"].map({"City": "#1f77b4", "Park": "#2ca02c"})

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.barh(plot_df["polygon_name"], plot_df["nni"], color=colors)
    ax.axvline(1.0, color="black", linewidth=1, linestyle="--", label="Random pattern (NNI = 1)")
    ax.set_xlabel("Nearest Neighbor Index")
    ax.set_ylabel("Polygon")
    ax.set_title("Flickr clustering by city and park polygon")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Simple static map for visual checking. GeoPandas is used only for plotting here.
if len(matched_points_df) > 0:
    cities_plot = gpd.read_file(GEOJSON_CITIES).to_crs(epsg=4326)
    parks_plot = gpd.read_file(GEOJSON_PARKS).to_crs(epsg=4326)

    plot_points_df = matched_points_df.drop_duplicates("PhotoID").copy()
    if len(plot_points_df) > 20000:
        plot_points_df = plot_points_df.sample(20000, random_state=42)

    fig, ax = plt.subplots(figsize=(9, 10))
    parks_plot.boundary.plot(ax=ax, color="#2ca02c", linewidth=1, label="Parks")
    cities_plot.boundary.plot(ax=ax, color="#1f77b4", linewidth=1, label="Cities")
    ax.scatter(
        plot_points_df["Longitude"],
        plot_points_df["Latitude"],
        s=2,
        alpha=0.25,
        color="#d62728",
        label="Matched Flickr photos"
    )
    ax.set_title("Matched Flickr photos inside selected German cities and parks")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

In [ ]:
print("Output files")
print(f"Compiled Flickr data: {COMPILED_CSV}")
print(f"Bounding-box candidates: {CANDIDATE_CSV}")
print(f"Point-in-polygon matches: {INSIDE_CSV}")
print(f"NNI results: {NNI_RESULTS_CSV}")